## Import Modules and Data

In [1]:
import pandas as pd
import RFE_Regression as rfe_regr
import data_analysis_utils as utils
from sklearn.preprocessing import StandardScaler



inp_dataset = pd.read_csv('kidney_disease.csv')
inp_dataset = pd.read_csv('kidney_disease.csv').drop(columns='id')

inp_dataset


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35,7300,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,...,47,6700,4.9,no,no,no,good,no,no,notckd
396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,...,54,7800,6.2,no,no,no,good,no,no,notckd
397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,...,49,6600,5.4,no,no,no,good,no,no,notckd
398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,...,51,7200,5.9,no,no,no,good,no,no,notckd


## Analysing Raw data and Preprocess

In [2]:
print('Missing Value Percentages:', inp_dataset.isnull().mean()*100)

quan, qual = utils.Preprocessing.quanQual(inp_dataset)
inp_dataset = utils.Preprocessing.quanVariables(inp_dataset, qual)
inp_dataset[['pcv', 'rc', 'wc']]=inp_dataset[['pcv', 'rc', 'wc']].astype(float)
quan, qual = utils.Preprocessing.quanQual(inp_dataset)
inp_dataset = utils.Preprocessing.simple_missing(inp_dataset, quan, qual)
inp_dataset = utils.Preprocessing.model_missing(inp_dataset, quan)


Missing Value Percentages: age                2.25
bp                 3.00
sg                11.75
al                11.50
su                12.25
rbc               38.00
pc                16.25
pcc                1.00
ba                 1.00
bgr               11.00
bu                 4.75
sc                 4.25
sod               21.75
pot               22.00
hemo              13.00
pcv               17.50
wc                26.25
rc                32.50
htn                0.50
dm                 0.50
cad                0.50
appet              0.25
pe                 0.25
ane                0.25
classification     0.00
dtype: float64


## Scaled and encoded Input, Target variables split

In [3]:
quan_data = inp_dataset[quan]
quan_scaler = StandardScaler()
quan_scaled = quan_scaler.fit_transform(quan_data)
scaled_data = pd.DataFrame(quan_scaled, columns=quan)
scaled_data[qual] = inp_dataset[qual]

data = pd.get_dummies(scaled_data, drop_first=True)
indep_X = data.iloc[:, :-1]
dep_Y = data.iloc[:, -1]


## Fearture Selection

In [4]:

# Step 1: Run RFE to reduce features (keeps DataFrame instead of NumPy arrays)

rfelist, features_list = rfe_regr.rfeFeature(indep_X, dep_Y, n_features=3)

# Step 2: Run classifiers on RFE outputs + include features
result = rfe_regr.rfe_regression(rfelist, features_list, dep_Y)

print("\nFinal R2 Score Table (with Features):")
result



Final R2 Score Table (with Features):


,No_of_Features,RFE_Model,Selected_Features,Logistic,SVMl,Decision,Random
0,3,LogisticRFE,"rbc_normal, pc_normal, htn_yes",0.386498,0.117622,0.484319,0.487289
1,3,SVCRFE,"rbc_normal, pc_normal, htn_yes",0.386498,0.117622,0.484319,0.479829
2,3,RandomForestRFE,"hemo, rbc_normal, htn_yes",0.538687,0.532470,0.815968,0.826622
3,3,DecisionTreeRFE,"hemo, rbc_normal, dm_yes",0.537110,0.528505,0.838768,0.826966


In [5]:
fullresult = pd.DataFrame()
for n in range(3,7):
    rfelist, features_list = rfe_regr.rfeFeature(indep_X, dep_Y, n_features=n)

    # Step 2: Run classifiers on RFE outputs + include features
    result = rfe_regr.rfe_regression(rfelist, features_list, dep_Y)
    fullresult = pd.concat([fullresult, result], ignore_index=True)


fullresult


,No_of_Features,RFE_Model,Selected_Features,Logistic,SVMl,Decision,Random
0,3,LogisticRFE,"rbc_normal, pc_normal, htn_yes",0.386498,0.117622,0.484319,0.484340
1,3,SVCRFE,"rbc_normal, pc_normal, htn_yes",0.386498,0.117622,0.484319,0.471658
2,3,RandomForestRFE,"hemo, rbc_normal, htn_yes",0.538687,0.532470,0.815968,0.824116
3,3,DecisionTreeRFE,"hemo, rbc_normal, dm_yes",0.537110,0.528505,0.838768,0.850512
4,4,LogisticRFE,"hemo, rbc_normal, pc_normal, htn_yes",0.582309,0.513109,0.947819,0.935204
5,4,SVCRFE,"rbc_normal, pc_normal, htn_yes, dm_yes",0.433430,0.202380,0.557504,0.562591
6,4,RandomForestRFE,"sg, hemo, rbc_normal, htn_yes",0.585559,0.583666,0.868083,0.896042
7,4,DecisionTreeRFE,"bu, hemo, rbc_normal, dm_yes",0.536001,0.521220,0.824111,0.876877
8,5,LogisticRFE,"hemo, rbc_normal, pc_normal, htn_yes, dm_yes",0.606439,0.552083,0.947819,0.904684
9,5,SVCRFE,"hemo, rbc_normal, pc_normal, htn_yes, dm_yes",0.606439,0.552083,0.947819,0.882109


In [6]:
max_val = fullresult.iloc[:, 3:].values.max()
row, col = fullresult.iloc[:, 3:].stack().idxmax()
no_features = fullresult.iloc[row, 0]
features = fullresult.iloc[row,2]
rfe_model = fullresult.iloc[row,1]
print(f'Max Accuracy is: {max_val:.4f}% for {rfe_model} Model and {col} Model  \nWith No of Features = {no_features}, Selected Features: {features}')


Max Accuracy is: 0.9625% for LogisticRFE Model and Decision Model  
With No of Features = 6, Selected Features: hemo, rbc_normal, pc_normal, ba_present, htn_yes, dm_yes
